In [7]:
import pandas as pd, numpy as np, joblib, re
from pathlib import Path
from sklearn.ensemble import HistGradientBoostingClassifier

table = pd.read_parquet("../data/processed/model_table.parquet")
FEATURES = [c for c in table if c not in ("cik", "end") and not c.startswith("y_")]
annee = table["end"].dt.year
table["split"] = np.select([annee <= 2019, annee <= 2021, annee <= 2024],
                           ["train", "val", "test"], default="prediction")

EVENEMENTS = {
    "Capitaux propres : perte > 10 % des actifs":
        (table["y_E"] < -0.10,                       table["y_E"].notna()),
    "Trésorerie divisée par 2 ou plus":
        (table["y_C"] < -0.5 * table["C_A"],         table["y_C"].notna() & (table["C_A"] > 0)),
    "Actifs en baisse de plus de 20 %":
        (table["y_A"] < np.log(0.8),                 table["y_A"].notna()),
    "Capitaux propres deviennent négatifs":
        (table["E_A"] + table["y_E"] < 0,            table["y_E"].notna() & (table["E_A"] > 0)),
}
COURT = {
    "Capitaux propres : perte > 10 % des actifs": "perte_CP",
    "Trésorerie divisée par 2 ou plus":           "treso_moitie",
    "Actifs en baisse de plus de 20 %":           "actifs_chute",
    "Capitaux propres deviennent négatifs":       "CP_negatifs",
}
# Meilleurs réglages trouvés sur la validation (cellule A du notebook 07)
PARAMS = {
    "perte_CP":     dict(learning_rate=0.03, max_leaf_nodes=31, min_samples_leaf=100),
    "treso_moitie": dict(learning_rate=0.10, max_leaf_nodes=31, min_samples_leaf=50),
    "actifs_chute": dict(learning_rate=0.05, max_leaf_nodes=31, min_samples_leaf=100),
    "CP_negatifs":  dict(learning_rate=0.03, max_leaf_nodes=15, min_samples_leaf=300),
}
# Poids du score : selon la gravité de l'événement pour une banque
POIDS = {"CP_negatifs": 0.35, "perte_CP": 0.25, "actifs_chute": 0.25, "treso_moitie": 0.15}

In [8]:
Path("../models").mkdir(exist_ok=True)
modeles = {}
for nom, (evt, ok) in EVENEMENTS.items():
    c = COURT[nom]
    tr = table[ok & table["split"].isin(["train", "val", "test"])]
    clf = HistGradientBoostingClassifier(max_iter=300, l2_regularization=1.0,
                                         random_state=0, **PARAMS[c])
    clf.fit(tr[FEATURES], evt[tr.index].astype(int))
    joblib.dump(clf, f"../models/prod_{c}.joblib")
    modeles[c] = clf
    print(f"{c:<14} entraîné sur {len(tr):,} bilans")

perte_CP       entraîné sur 71,650 bilans
treso_moitie   entraîné sur 69,693 bilans
actifs_chute   entraîné sur 78,863 bilans
CP_negatifs    entraîné sur 63,954 bilans


In [9]:
def scorer(df, modeles):
    out = df[["cik", "end"]].copy()
    out["annee"] = df["end"].dt.year
    for c, m in modeles.items():
        out[f"p_{c}"] = m.predict_proba(df[FEATURES])[:, 1]                    # probabilité
        out[f"rang_{c}"] = out.groupby("annee")[f"p_{c}"].rank(pct=True) * 100  # rang dans l'année
    brut = sum(w * out[f"rang_{c}"] for c, w in POIDS.items())                  # moyenne pondérée
    out["score"] = brut.groupby(out["annee"]).rank(pct=True) * 100             # NOUVEAU : re-classé de 0 à 100
    out["niveau"] = pd.cut(out["score"], [0, 50, 75, 90, 100.01], right=False,
                           labels=["Faible", "Modéré", "Élevé", "Très élevé"])
    return out

In [10]:
anciens = {COURT[nom]: joblib.load(f"../models/{nom[:25].replace(' ', '_').replace(':', '')}.joblib")
           for nom in EVENEMENTS}
te = table[table["split"] == "test"]
s = scorer(te, anciens)
for nom, (evt, ok) in EVENEMENTS.items():
    s[COURT[nom]] = evt[te.index].astype(float).where(ok[te.index])

verif = s.groupby("niveau", observed=True)[list(COURT.values())].mean()
verif.insert(0, "entreprises", s.groupby("niveau", observed=True).size())
verif.round(3)

,entreprises,perte_CP,treso_moitie,actifs_chute,CP_negatifs
niveau,,,,,
Faible,8474,0.042,0.097,0.022,0.003
Modéré,4240,0.287,0.160,0.198,0.049
Élevé,2543,0.467,0.275,0.401,0.140
Très élevé,1697,0.495,0.305,0.392,0.158


In [11]:
SRC = Path("E:/DSI/projet analytics edge")
noms = {}
for f in SRC.glob("CIK*.json"):
    with open(f, "rb") as fh:
        tete = fh.read(600).decode("utf-8", errors="ignore")    # le nom est au début du fichier
    m = re.search(r'"entityName"\s*:\s*"([^"]*)"', tete)
    if m:
        noms[f.stem[3:]] = m.group(1)
print(f"{len(noms):,} noms trouvés")

20,526 noms trouvés


In [12]:
scores = scorer(table, modeles)
scores["nom"] = scores["cik"].map(noms).fillna("(nom inconnu)")

RATIOS = ["taille", "L_A", "E_A", "C_A", "RE_A", "AC_LC", "ROA", "marge_op",
          "CFO_A", "couv_interets", "d_CA"]
scores = scores.join(table[RATIOS])
scores.to_parquet("../data/processed/scores.parquet", index=False)

print(scores["niveau"].value_counts())
scores.sort_values(["annee", "score"], ascending=[False, False]).head(10)

niveau
Faible        46399
Modéré        23213
Élevé         13924
Très élevé     9295
Name: count, dtype: int64


,cik,end,annee,p_perte_CP,rang_perte_CP,p_treso_moitie,rang_treso_moitie,p_actifs_chute,rang_actifs_chute,p_CP_negatifs,...,L_A,E_A,C_A,RE_A,AC_LC,ROA,marge_op,CFO_A,couv_interets,d_CA
63105,0001434601,2026-05-31,2026,0.625967,98.668639,0.387030,95.710059,0.295282,88.461538,0.598611,...,2.904521,-1.904521,0.060327,-7.887012,0.026185,-2.124496,-0.927195,-0.506458,NaN,NaN
90101,0001858007,2026-03-31,2026,0.379155,82.544379,0.506331,99.556213,0.661395,99.704142,0.607157,...,2.286251,-1.332372,NaN,-1.332403,0.001718,-0.069567,NaN,-0.041750,-17.814260,NaN
70817,0001527728,2026-06-30,2026,0.452791,88.461538,0.458127,98.668639,0.360002,93.047337,0.518689,...,1.907779,-0.907779,0.172369,-51.703481,0.107686,-0.358334,-167.293680,-0.531916,NaN,NaN
35535,0001040130,2026-03-31,2026,0.563575,96.449704,0.351861,94.378698,0.459759,98.224852,0.154021,...,0.643557,0.356443,0.263724,0.114089,0.837222,-0.705571,-0.327744,-0.349719,NaN,-0.237324
83850,0001729637,2026-05-31,2026,0.456543,89.349112,0.300098,90.236686,0.305193,88.905325,0.545263,...,1.622196,-0.622196,NaN,-2.659715,0.600063,-1.412971,-0.198093,-0.817371,-3.331827,2.871089
45603,0001158780,2026-06-30,2026,0.448847,87.573964,0.257195,85.798817,0.366278,94.230769,0.525586,...,1.925959,-0.925959,0.141852,-18.766621,0.266817,-0.957513,-24.545276,-0.787362,NaN,-0.273807
83041,0001711012,2026-03-31,2026,0.495631,91.420118,0.214477,77.810651,0.394401,95.266272,0.507517,...,1.479208,-1.109502,NaN,-9.582135,0.526175,-1.002382,-1.868021,-0.304614,NaN,-0.784803
27535,0000911216,2026-06-30,2026,0.593954,97.928994,0.356390,94.526627,0.376556,94.822485,0.086838,...,0.215558,0.784442,0.878947,-54.840427,4.607569,-0.982876,-0.981686,-1.587694,NaN,NaN
69458,0001512922,2026-03-31,2026,0.458755,89.497041,0.395405,96.449704,0.357700,92.899408,0.262360,...,0.385069,0.614931,NaN,-28.538763,1.350419,-2.928272,-7.938776,-1.707500,-715.478172,0.007980
62940,0001433551,2026-06-30,2026,0.530096,94.378698,0.381203,95.414201,0.139947,73.816568,0.660173,...,1.556328,-0.556328,0.044765,-1.337016,0.099233,-0.305650,-0.858091,-0.021865,NaN,0.060981


## Synthèse : Score de risque
- Score = moyenne pondérée des rangs des 4 probabilités (poids : CP négatifs 0,35 ; perte CP 0,25 ; actifs −20 % 0,25 ; trésorerie 0,15),
  puis re-classé de 0 à 100 dans chaque année → « plus risquée que X % des entreprises de l'année ».
- Niveaux : Faible (< 50), Modéré (50-75), Élevé (75-90), Très élevé (≥ 90).
- Vérification sur le test (modèles ≤ 2021) : taux de CP négatifs 0,3 % → 4,9 % → 14,0 % → 15,8 % ; actifs −20 % : 2,2 % → 19,8 % → 40,1 % → 39,2 %.
- Limite : l'agrégation dilue le signal le plus grave → le dashboard affiche aussi les 4 alertes détaillées.
- Modèles de production entraînés sur toutes les années ≤ 2024 (models/prod_*.joblib).